# LIFE on Google Colab — GossipCop++ (binary MF-vs-MR, LLaMA2-7B)

Faithful reproduction of the paper's setup: binary fake/real over the **LLM pair** (MF=fake, MR=real), with **LLaMA2-7B** as the reconstruction model. End-to-end: **convert → key-sentence extraction → concatenate → features → train**.

**Before you start:**
1. Set the Colab runtime to **GPU** — an **A100** is needed for LLaMA2-7B (Runtime → Change runtime type).
2. Upload the whole `LIFE` repo (including `dataset/data/`) to your Google Drive, e.g. `MyDrive/LIFE`.
3. Edit `PROJECT_DIR` in the path cell below if you put it somewhere else.
4. Step 3 uses the **ungated** `NousResearch/Llama-2-7b-hf` mirror by default — no HF token needed. (The official `meta-llama/Llama-2-7b-hf` is gated and requires an approved access request + token.)

Scope: **GossipCop++** only (~8253 LLM-pair articles: 4084 fake + 4169 real). VLPFN is excluded (its text has no punctuation, so sentence splitting cannot work). GossipCop++ is far heavier; try it only after this works.

In [12]:
# Confirm a GPU is attached
!nvidia-smi

Wed Jun 10 21:25:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [13]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
import os

# <-- change this if you uploaded the repo elsewhere
PROJECT_DIR = '/content/drive/MyDrive/LIFE'
os.chdir(PROJECT_DIR)

GOSSIPCOP_DIR = f'{PROJECT_DIR}/dataset/data/Fakenews-dataset-main/Fakenews-dataset-main/Dataset/GossipCop++'

# Paper's binary task: LLM pair only (MF=fake, MR=real), reconstructed with LLaMA2-7B.
OUTPUT_BIN     = f'{PROJECT_DIR}/dataset/output_bin'        # MF_fake.jsonl + MR_true.jsonl
KEY_SENT       = f'{PROJECT_DIR}/dataset/keySentence/important_sentences_top15.jsonl'
BERT_CKPT      = f'{PROJECT_DIR}/dataset/bert_bin.pt'       # fresh extractor for MF-vs-MR
FEATURES_LLAMA = f'{PROJECT_DIR}/dataset/features_llama'
TRAIN_PATH     = f'{PROJECT_DIR}/dataset/train_bin.jsonl'
TEST_PATH      = f'{PROJECT_DIR}/dataset/test_bin.jsonl'

print('cwd:', os.getcwd())
print('GossipCop++ found:', os.path.isdir(GOSSIPCOP_DIR))

cwd: /content/drive/MyDrive/LIFE
GossipCop++ found: True


In [15]:
# Install dependencies.
# If the fastNLP import fails at the training step, pin a compatible version:
#   !pip install -q fastNLP==1.0.1
!pip install -q -r requirements.txt

In [16]:
# NLTK sentence tokenizer data. 'punkt' gives english.pickle (used by train.py);
# 'punkt_tab' is required by newer nltk's sent_tokenize (used by step 1).
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## HuggingFace login (optional)
Step 3 defaults to the **ungated** `NousResearch/Llama-2-7b-hf` mirror, so **no token is needed — you can skip this cell**. Only run it if you switch Step 3 to the official gated `meta-llama/Llama-2-7b-hf` (which also requires an approved access request).

In [17]:
# LLaMA-2 is gated on HuggingFace. First accept the license at
# https://huggingface.co/meta-llama/Llama-2-7b-hf, then run this cell and paste an
# access token from https://huggingface.co/settings/tokens
# (or replace with: login(token="hf_xxx")).
# from huggingface_hub import login
# login()

## Step 0 — Convert GossipCop++ to the binary LLM-pair JSONL
`--subset llm` emits only `MF_fake.jsonl` (4084, fake) and `MR_true.jsonl` (4169, real) — the paper's binary task. HF/HR (human-written) are not used.

In [18]:
!python dataset/0_convert.py --input_dir "{GOSSIPCOP_DIR}" --output_dir "{OUTPUT_BIN}" --subset llm

MF.json -> MF_fake.jsonl: 4084 records (label=gpt3.5_fake)
MR.json -> MR_true.jsonl: 4169 records (label=gpt3.5_true)


## Step 1 — Key-sentence extraction (top-15)
Trains a **fresh** BERT fake/real classifier on MF-vs-MR (saved to `BERT_CKPT`), then keeps the **top-15** most impactful sentences per article (paper's k for GossipCop++). This is the slowest step (a forward pass per sentence per article).

In [19]:
!python dataset/1_keySentenceExtraction.py --data_dir "{OUTPUT_BIN}" --output_file "{KEY_SENT}" --top_k 15 --model_path "{BERT_CKPT}" --gpu 0

Loading weights: 100% 199/199 [00:00<00:00, 4416.18it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider

## Step 2 — Concatenate key sentences back into the data
Adds a `sentence` field to each record in `OUTPUT_BIN` by matching on `(id, label)`. **Overwrites the files in `OUTPUT_BIN` in place** — re-run Step 0 first if you need to reset them.

In [20]:
!python dataset/2_concate.py --folder_path "{OUTPUT_BIN}" --important_sentences_file "{KEY_SENT}"

所有 .jsonl 文件已成功更新。


## Step 3 — Reconstruction probabilities with LLaMA2-7B
The paper's reconstruction model. A malicious prompt is prepended and LLaMA2-7B's per-token log-likelihoods over the key fragments form the "linguistic fingerprint" features. Loads in bfloat16 (~14 GB; needs the A100) and downloads ~13 GB on first run. Writes one feature JSONL per input file into `FEATURES_LLAMA`.

In [21]:
# meta-llama/Llama-2-7b-hf is gated (needs Meta approval). NousResearch/Llama-2-7b-hf is an
# ungated mirror of the SAME weights/tokenizer — no token needed. Swap back to the official
# repo if/when your access request is approved.
!python dataset/3_gen_features_local.py --input_dir "{OUTPUT_BIN}" --output_dir "{FEATURES_LLAMA}" --model NousResearch/Llama-2-7b-hf --scorer llama --dtype bfloat16 --gpu 0

Streaming output truncated to the last 5000 lines.
816 840
910 954
956 994
 92% 3819/4169 [05:07<00:30, 11.65it/s]0 117
118 176
178 226
228 258
261 299
301 335
337 369
372 422
424 432
434 452
454 478
480 530
533 589
591 653
656 704
706 762
0 117
118 132
134 186
189 231
261 309
312 374
377 419
421 485
560 570
593 647
649 689
692 748
750 818
890 930
932 1004
1006 1044
 92% 3821/4169 [05:08<00:29, 11.77it/s]0 57
58 70
0 117
118 180
182 214
216 276
279 347
376 418
493 547
549 585
606 644
646 656
719 767
770 782
817 861
863 891
894 910
928 958
 92% 3823/4169 [05:08<00:29, 11.73it/s]0 117
118 156
158 202
205 253
255 277
279 317
320 344
346 406
408 444
499 535
538 562
564 592
594 630
633 679
714 752
754 790
0 117
118 174
215 261
303 335
337 361
364 364
366 410
412 444
447 447
449 497
530 530
532 574
576 624
627 627
629 671
823 867
 92% 3825/4169 [05:08<00:28, 11.88it/s]0 117
206 256
259 284
286 328
331 331
333 389
392 392
394 400
445 445
447 447
492 492
494 536
539 583
585 617
620 662
664 698

## Step 4 — Train the classifier (binary)
Splits `FEATURES_LLAMA` into train/test and trains the Transformer+CRF classifier for **15 epochs** on the binary MF-vs-MR task. Paper target for GossipCop++: **Acc 0.937 / F1 0.924**.

In [22]:
!python LIFE_train/train.py \
  --split_dataset \
  --data_path "{FEATURES_LLAMA}" \
  --train_path "{TRAIN_PATH}" \
  --test_path "{TEST_PATH}" \
  --model Transformer \
  --num_train_epochs 15

Log INFO: split dataset...
********************************
The overall data sources:
['MF_fake.jsonl', 'MR_true.jsonl']
100% 6602/6602 [00:02<00:00, 3151.60it/s]
100% 1651/1651 [00:00<00:00, 3176.91it/s]

The number of train dataset: 6602
The number of test  dataset: 1651
********************************
100% 6602/6602 [00:04<00:00, 1633.87it/s]
100% 1651/1651 [00:00<00:00, 7252.51it/s]
--------------------------------classify--------------------------------
Log INFO: do train...
Epoch:   0% 0/15 [00:00<?, ?it/s]
Iteration:   0% 0/207 [00:00<?, ?it/s]
Iteration:   0% 1/207 [00:01<03:44,  1.09s/it]
Iteration:   1% 2/207 [00:01<01:47,  1.91it/s]
Iteration:   1% 3/207 [00:01<01:09,  2.93it/s]
Iteration:   2% 4/207 [00:01<00:51,  3.93it/s]
Iteration:   2% 5/207 [00:01<00:41,  4.87it/s]
Iteration:   3% 6/207 [00:01<00:35,  5.69it/s]
Iteration:   3% 7/207 [00:01<00:31,  6.40it/s]
Iteration:   4% 8/207 [00:01<00:28,  6.98it/s]
Iteration:   4% 9/207 [00:02<00:26,  7.39it/s]
Iteration:   5% 10

## Notes / troubleshooting
- **HF gating**: Step 3 defaults to the ungated `NousResearch/Llama-2-7b-hf` mirror (no token). If you switch to the official `meta-llama` repo and hit a 403 "gated repo", your access request hasn't been approved yet.
- **fastNLP**: if Step 4 errors on `from fastNLP.modules.torch import ...`, run `!pip install -q fastNLP==1.0.1` and restart the runtime.
- **Checkpoints**: `BERT_CKPT` (step 1) and `linear_en.pt` (step 4) are written under `PROJECT_DIR` on Drive, so they survive disconnects.
- **NaN features**: if Step 3 prints NaN/inf, switch Step 3 to `--dtype float32` (fits the 40 GB A100).
- **Re-runs**: Step 2 mutates `OUTPUT_BIN` in place; always re-run Step 0 before re-running Steps 1–3 from scratch.